## hook
- 在某个既定流程的特定时机，被框架、系统或主程序自动调用的扩展函数,分为两类6种:
    - Node-style hooks(节点风格钩子)---两种方式:类注解(底层还是创建继承AgentMiddleware的类)/创建继承AgentMiddleware类,重写before_agent,before_model,after_model,after_agent方法
        - before_agent
        - before_model
        - after_model
        - after_agent
    - Wrap-style hooks(包装风格钩子),同上两种方式,可以对request参数进行修改
        - wrap_tool_call
        - wrap_model_call
- 装饰器写法更适合单个hook、逻辑简单、快速原型的场景；
- 类写法更适合多个 hook 组合、复杂配置、需要同时提供同步/异步实现、以及更强复用与可测试性的场景
- 执行顺序:
    - 1.中间件定义是乱序的，但传递给Agent的顺序是固定的
    - 2.由输出可知，中间件的执行遵循上述规律,只和传递给Agent的顺序有关
    - 3.具体来说:
        - before_model中间件的执行顺序和传递顺序一致
        - after_model中间件的执行顺序和传递顺序相反
        - wrap_model_call中间件的执行顺序是： 先传递的包在最外层 ，即洋葱架构



In [15]:
from dotenv import load_dotenv
from typing import Any

from langchain_core.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_agent, after_model, before_model, after_agent, hook_config, \
    wrap_model_call
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    #思考模式下不支持调用工具
    extra_body = {"thinking":{"type":"disabled"}}
)

@before_agent(can_jump_to=["end"])#可以跳转到end/model/tools节点(注意:这里只是声明有能力,return {"jump_to": "end"})才是真正跳转
def before_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """在智能体执行开始前调用"""
    state["messages"][-1].content += "<before_agent>"
    return None

@before_model(can_jump_to=["end"])#可以跳转到end/model/tools节点
def before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """在模型执行开始前调用"""
    state["messages"][-1].content += "<before_model>"
    return None

@after_model(can_jump_to=["end"])#可以跳转到end/model/tools节点
def after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """在模型执行完成后调用"""
    state["messages"][-1].content += "<after_model>"
    return {"jump_to": "end"}

@after_agent
def after_agent(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """在智能体执行完成后调用"""
    state["messages"][-1].content += "<after_agent>"
    return None

agent = create_agent(
    model = model,
    middleware=[
        before_agent,
        before_model,
        after_agent,
        after_model,
    ],
)

response = agent.invoke({"messages":[{"role":"user","content":"今天星期几"}]})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

今天星期几<before_agent><before_model>
================================== Ai Message ==================================

让我帮你查看今天的日期。不过我需要说明一下，我的知识截止日期是2025年5月，没有实时联网功能。

如果你想获取当前的准确日期，可以：

1. **查看你设备上的日历或时钟**
2. **打开联网搜索** - 如果你在使用支持联网搜索的界面，可以开启该功能后重新问我
3. **告诉我你的具体时间** - 如果你知道日期，我可以帮你推算星期几

如果你想要我推算，请告诉我今天的**年、月、日**（例如“今天是2025年6月15日”），我可以帮你算出星期几。

如果你目前无法确认日期，也可以告诉我你现在所处的环境（比如手机、电脑），我可以给出通用的查看方法。

你希望怎么处理呢？<after_model><after_agent>


In [5]:
from dotenv import load_dotenv
from typing import Any
from langgraph.runtime import Runtime
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import  AgentMiddleware
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    #思考模式下不支持调用工具
    extra_body = {"thinking":{"type":"disabled"}}
)

class MyMiddleware(AgentMiddleware):
    """自定义hook"""
    # def __init__(self):
    #     super().__init__()只有需要初始化自己才需要,否则可以不写
    @hook_config(can_jump_to=["end"])
    def before_agent(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        """在智能体执行开始前调用"""
        state["messages"][-1].content += "<before_agent>"
        return {"jump_to": "end"}

    def before_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        """在模型执行开始前调用"""
        state["messages"][-1].content += "<before_model>"
        return None

    def after_model(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        """在模型执行完成后调用"""
        state["messages"][-1].content += "<after_model>"
        return None

    def after_agent(self,state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        """在智能体执行完成后调用"""
        state["messages"][-1].content += "<after_agent>"
        return None

agent = create_agent(
    model = model,
    middleware=[
        MyMiddleware(),
    ],
)

response = agent.invoke({"messages":[{"role":"user","content":"你好"}]})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

你好<before_agent><before_model>
================================== Ai Message ==================================

你好！😊 很高兴见到你！有什么我可以帮忙的吗？无论是回答问题、提供信息、协助解决问题，还是陪你聊聊天，我都在这儿呢～随时告诉我你的需求吧！<after_model><after_agent>


In [20]:
from typing import Callable
from langchain.agents.middleware import ModelRequest, ModelResponse
from langchain.agents.middleware.types import wrap_model_call
from langchain_core.messages import HumanMessage
load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    #思考模式下不支持调用工具
    extra_body = {"thinking":{"type":"disabled"}}
)

@wrap_model_call
#request为模型请求,handler为模型处理函数
def wrap_model_call_middle(request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse]
    ) -> ModelResponse:
    request=request.override(messages=[HumanMessage("1+1=?")])
    return handler(request)

agent = create_agent(
    model = model,
    middleware=[
        wrap_model_call_middle,
    ],
)

response = agent.invoke({"messages":[{"role":"user","content":"你好"}]})

for msg in response["messages"]:
    msg.pretty_print()



================================ Human Message =================================

你好
================================== Ai Message ==================================

1 + 1 = **2**.


In [21]:
from typing import Callable
from langchain.agents.middleware import ModelRequest, ModelResponse
from langchain.agents.middleware.types import wrap_model_call
from langchain_core.messages import HumanMessage
load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    #思考模式下不支持调用工具
    extra_body = {"thinking":{"type":"disabled"}}
)

class MyMiddleware(AgentMiddleware):
    """自定义hook"""

    def wrap_model_call(self,request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse]
        ) -> ModelResponse:
        request=request.override(messages=[HumanMessage("1+1=?")])
        return handler(request)

agent = create_agent(
    model = model,
    middleware=[
        MyMiddleware(),
    ],
)

response = agent.invoke({"messages":[{"role":"user","content":"你好"}]})

for msg in response["messages"]:
    msg.pretty_print()



================================ Human Message =================================

你好
================================== Ai Message ==================================

In common arithmetic, **1 + 1 = 2**.  

However, depending on the context:
- In **binary** (base-2), 1 + 1 = **10** (which is "2" in decimal).
- In **Boolean algebra**, 1 + 1 = **1** (since OR is used, not addition).
- In a philosophical or trick-question sense, some might say 1 + 1 = **11** (if you just place the numbers side by side).  

But mathematically, the standard, universally accepted answer is **2**.
